LOADING THE DATA 

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

path = Path("../data_catalog/data_raw/")

dengue = pd.read_csv(path / "dengue.csv.gz", compression="gzip")
climate = pd.read_csv(path / "climate.csv.gz", compression="gzip")
forecast = pd.read_csv(path / "forecasting_climate.csv.gz", compression="gzip")
pop = pd.read_csv(path / "datasus_population_2001_2025.csv.gz", compression="gzip")
ocean = pd.read_csv(path / "ocean_climate_oscillations.csv.gz", compression="gzip")
env = pd.read_csv(path / "environ_vars.csv.gz", compression="gzip")

shape_muni = gpd.read_file(path / "shape_muni.gpkg")


In [3]:
shape_regional = pd.read_csv(path / "map_regional_health.csv")
shape_regional = shape_regional[['uf_code', 'uf']].drop_duplicates()
shape_regional = shape_regional.rename(columns={"uf": "state"})
shape_regional

,uf_code,state
0,12,AC
22,27,AL
124,13,AM
186,16,AP
202,29,BA
619,23,CE
803,53,DF
804,32,ES
882,52,GO
930,21,MA


CHECKING IF ALL THE WEEKS HAVE ALL THE UF AND IF THE UF HAVE ALL THE WEEKS 

In [4]:
def validate_spatiotemporal_panel(df, time_col, spatial_col, date_filter_col=None, date_min=None):
    data = df.copy()

    # filte data to only after begining of dengue dataset
    data = data[data[date_filter_col] >= date_min]

    # check how many unique spatial and time and min max time
    print(f"n_{spatial_col}s:", data[spatial_col].nunique())
    print(f"n_{time_col}s:", data[time_col].nunique())
    print("min:", data[date_filter_col].min())
    print("max:", data[date_filter_col].max())

    # checks if all the spatial units have the same number of time points and if all time points have the same number of spatial units
    spatial_uniform = data.groupby(spatial_col)[time_col].nunique().nunique() == 1
    temporal_uniform = data.groupby(time_col)[spatial_col].nunique().nunique() == 1

    print("uniform time per spatial unit:", spatial_uniform)
    print("uniform spatial units per time:", temporal_uniform)

    # checks for duplicates
    dup_check = data.duplicated([spatial_col, time_col]).any()
    print("has duplicates:", dup_check)

    # checks for completeness
    expected = data[spatial_col].nunique() * data[time_col].nunique()
    actual = data.groupby([spatial_col, time_col]).ngroups

    print("panel complete:", expected == actual)

    expected_index = pd.MultiIndex.from_product(
        [data[spatial_col].unique(), data[time_col].unique()],
        names=[spatial_col, time_col])

    actual_index = pd.MultiIndex.from_frame(data[[spatial_col, time_col]].drop_duplicates())

    missing = expected_index.difference(actual_index)

    print("missing pairs:", len(missing))

    # count missing values per column
    print(data.isna().sum())

In [5]:
validate_spatiotemporal_panel(
    dengue,
    time_col="epiweek",
    spatial_col="geocode",
    date_filter_col="date",
    date_min=dengue["date"].min())

n_geocodes: 5570
n_epiweeks: 845
min: 2010-01-03
max: 2026-03-08
uniform time per spatial unit: True
uniform spatial units per time: True
has duplicates: False
panel complete: True
missing pairs: 0
geocode                  0
date                     0
casos                    0
epiweek                  0
uf                       0
macroregional_geocode    0
regional_geocode         0
uf_code                  0
target_city              0
train_1                  0
target_1                 0
train_2                  0
target_2                 0
train_3                  0
target_3                 0
train_4                  0
target_4                 0
disease                  0
dtype: int64


In [6]:
validate_spatiotemporal_panel(
    climate,
    time_col="epiweek",
    spatial_col="geocode",
    date_filter_col="date",
    date_min=dengue["date"].min())

n_geocodes: 5567
n_epiweeks: 846
min: 2010-01-03
max: 2026-03-15
uniform time per spatial unit: True
uniform spatial units per time: True
has duplicates: False
panel complete: True
missing pairs: 0
date             0
epiweek          0
geocode          0
temp_min         0
temp_med         0
temp_max         0
precip_min       0
precip_med       0
precip_max       0
pressure_min     0
pressure_med     0
pressure_max     0
rel_humid_min    0
rel_humid_med    0
rel_humid_max    0
thermal_range    0
rainy_days       0
dtype: int64


In [7]:
validate_spatiotemporal_panel(
    forecast,
    time_col="reference_month",
    spatial_col="geocode",
    date_filter_col="reference_month",
    date_min=dengue["date"].min())

n_geocodes: 5570
n_reference_months: 194
min: 2010-02-01
max: 2026-03-01 00:00:00
uniform time per spatial unit: True
uniform spatial units per time: True
has duplicates: True
panel complete: True
missing pairs: 0
geocode                     0
reference_month             0
forecast_months_ahead       0
temp_med                 1074
umid_med                 1074
precip_tot               1074
dtype: int64


In [8]:
missing_forecast = forecast[forecast.isna().any(axis=1)]
missing_forecast.geocode.nunique()

1

In [9]:
g = forecast[forecast["geocode"] == missing_forecast["geocode"].iloc[0]]

print("NAs:", g.isna().any(axis=1).any())
print("Non NAS :", (~g.isna().any(axis=1)).any())

NAs: True
Non NAS : True


In [10]:
forecast = forecast[~forecast.geocode.isin(missing_forecast.geocode.unique())]

In [11]:
year_min = pd.to_datetime(dengue["date"].min()).year
validate_spatiotemporal_panel(
    pop,
    time_col="year",
    spatial_col="geocode",
    date_filter_col="year",
    date_min=year_min)

n_geocodes: 5571
n_years: 16
min: 2010
max: 2025
uniform time per spatial unit: False
uniform spatial units per time: False
has duplicates: False
panel complete: False
missing pairs: 15
geocode       0
year          0
population    0
dtype: int64


In [12]:
geo_pop = set(pop["geocode"].unique())
dengue_pop = set(dengue["geocode"].unique())

geo_unknown = geo_pop - dengue_pop

pop[pop["geocode"].isin(geo_unknown)]

,geocode,year,population
135177,5101837,2025,5877


In [13]:
pop = pop[~pop["geocode"].isin(geo_unknown)]
year_min = pd.to_datetime(dengue["date"].min()).year
validate_spatiotemporal_panel(
    pop,
    time_col="year",
    spatial_col="geocode",
    date_filter_col="year",
    date_min=year_min)

n_geocodes: 5570
n_years: 16
min: 2010
max: 2025
uniform time per spatial unit: True
uniform spatial units per time: True
has duplicates: False
panel complete: True
missing pairs: 0
geocode       0
year          0
population    0
dtype: int64


In [14]:
ocean["date"] = pd.to_datetime(ocean["date"])
ocean = ocean[(ocean["date"] >= dengue["date"].min())]


ocean["week"] = ocean["date"] - pd.to_timedelta(ocean["date"].dt.weekday + 1, unit="D")

obs = ocean["week"].unique()
full = pd.date_range(ocean["week"].min(), ocean["week"].max(), freq="W-SUN")

print("missing weeks:", len(set(full) - set(obs)))
print("Values per week :", ocean["week"].value_counts().max())

print(ocean["date"].dt.day_name().value_counts())

missing weeks: 0
Values per week : 1
date
Sunday    854
Name: count, dtype: int64


PROBLEMS:
- Climate has missing geocodes - It should be fine as I'm averaging to STATE level
- Forecast - one geocode has missing values but also with values - I'll just eliminate and do the same as above 
- Ocean df with weird dates - but shifted everything to the sunday before- Will still miss 5 dates but I'll interpolate
- Population is somehow solved as irrelevant numbers 



In [15]:
uf_week = (dengue.groupby(["uf_code", "epiweek"], as_index=False).agg({
        "casos": "sum",
        "train_1": "first",
        "target_1": "first",
        "train_2": "first",
        "target_2": "first",
        "train_3": "first",
        "target_3": "first",
        "train_4": "first",
        "target_4": "first"}))

uf_week = uf_week.rename(columns={"casos": "cases"})

In [16]:
#calculate the area of each geocode (municipality as after we will have to use it to do weighted average of the climate variables
gdf = shape_muni.copy()

gdf = gdf.to_crs(epsg=5880)

gdf["area"] = gdf.geometry.area

gdf = gdf[['geocode', 'area', 'uf_code', 'uf']]

In [17]:
gdf

,geocode,area,uf_code,uf
0,1100015,7.138716e+09,11,RO
1,1100023,4.480712e+09,11,RO
2,1100031,1.322752e+09,11,RO
3,1100049,3.821895e+09,11,RO
4,1100056,2.804548e+09,11,RO
...,...,...,...,...
5565,5222005,9.583356e+08,52,GO
5566,5222054,7.354806e+08,52,GO
5567,5222203,1.059823e+09,52,GO
5568,5222302,2.189991e+09,52,GO


In [18]:
climate = climate[(climate["date"] >= dengue["date"].min())]

In [19]:
climate_uf = climate.merge(
    gdf[["geocode", "area", 'uf', 'uf_code']],
    on="geocode",
    how="left")



In [20]:
climate_uf["date"] = pd.to_datetime(climate_uf["date"], format="mixed")

In [21]:
non_climate =  ['date', 'epiweek', 'geocode', 'uf_area_share', 'uf', 'uf_code', 'year_month', 'reference_month', 'area']
value_cols = [col for col in climate_uf.columns.tolist() if col not in non_climate]

In [22]:
group_cols = ["date", "epiweek", "uf_code"]

weights = climate_uf["area"] / climate_uf.groupby(group_cols)["area"].transform("sum")

climate_uf = (
    climate_uf
    .assign(**{c: climate_uf[c] * weights for c in value_cols})
    .groupby(group_cols, as_index=False)
    .agg({c: "sum" for c in value_cols}))

In [23]:
climate_uf["year_month"] = (climate_uf["date"].dt.strftime("%Y%m").astype(int))

In [24]:
climate_uf

,date,epiweek,uf_code,temp_min,temp_med,temp_max,precip_min,precip_med,precip_max,pressure_min,pressure_med,pressure_max,rel_humid_min,rel_humid_med,rel_humid_max,thermal_range,rainy_days,year_month
0,2010-01-03,201001,11,23.405395,25.741168,29.050873,1.064855,23.107950,66.713673,0.969816,0.972566,0.974597,72.999984,87.842121,96.886100,5.645478,7.000000,201001
1,2010-01-03,201001,12,23.444156,26.308171,30.248567,0.677039,9.421365,35.514987,0.965062,0.968027,0.970210,64.072237,81.682621,93.912157,6.804412,7.000000,201001
2,2010-01-03,201001,13,23.758244,26.074562,29.339151,3.285306,26.361981,74.048928,0.981700,0.984253,0.986303,72.223857,86.909950,96.182576,5.580907,7.000000,201001
3,2010-01-03,201001,14,22.885245,26.370853,30.688115,0.643011,6.043738,18.992403,0.967684,0.970036,0.972078,51.253556,68.928538,82.458877,7.802871,5.805488,201001
4,2010-01-03,201001,15,22.868740,25.406705,28.805698,4.522010,28.134661,73.511111,0.972519,0.974751,0.976496,69.079149,84.651512,95.072615,5.936958,7.000000,201001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22837,2026-03-15,202611,43,20.010969,23.821824,28.524775,0.622754,6.114450,15.115265,0.963623,0.964785,0.966094,58.181128,76.405617,90.206544,8.513807,2.425091,202603
22838,2026-03-15,202611,50,23.785919,27.080869,31.038376,0.321111,3.568668,11.271656,0.956928,0.959348,0.961273,60.987021,75.207611,86.573308,7.252457,2.945187,202603
22839,2026-03-15,202611,51,22.907802,25.516029,29.186879,1.308828,10.442840,29.566565,0.956192,0.959103,0.961254,69.941424,86.125892,95.947271,6.279077,3.000000,202603
22840,2026-03-15,202611,52,21.198527,24.213874,28.171538,1.405339,8.566292,22.117157,0.926190,0.928380,0.930269,65.117423,82.745350,95.062527,6.973011,3.000000,202603


In [25]:
# as the df of climate has different number of geocodes of dengue and of forecast, the weighted average will be done separately and then merge the two


In [26]:
#forecast has a column called forecast_months_ahead and makes the identifier of each row the 3 variables - geocode, month and forecast_months_ahead
# in order to have the identifier the pair geocode and month, we will pivot that variable

forecast["reference_month"] = pd.to_datetime(forecast["reference_month"],format="mixed")

forecast["year_month"] = (forecast["reference_month"].dt.strftime("%Y%m").astype(int))

forecast = forecast.rename(columns={
	"precip_tot": "forecast_precip_tot",
    "temp_med": "forecast_temp_med",
    "umid_med": "forecast_umid_med",})

forecast_wide = (
    forecast
    .pivot(
        index=["geocode", "reference_month", "year_month"],
        columns="forecast_months_ahead",
        values=[
            "forecast_precip_tot",
            "forecast_temp_med",
            "forecast_umid_med"]))

forecast_wide.columns = [f"{metric}_{h}m"for metric, h in forecast_wide.columns]

forecast = forecast_wide.reset_index()
forecast



,geocode,reference_month,year_month,forecast_precip_tot_1m,forecast_precip_tot_2m,forecast_precip_tot_3m,forecast_precip_tot_4m,forecast_precip_tot_5m,forecast_precip_tot_6m,forecast_temp_med_1m,...,forecast_temp_med_3m,forecast_temp_med_4m,forecast_temp_med_5m,forecast_temp_med_6m,forecast_umid_med_1m,forecast_umid_med_2m,forecast_umid_med_3m,forecast_umid_med_4m,forecast_umid_med_5m,forecast_umid_med_6m
0,1100015,2010-01-01,201001,0.000117,0.000112,0.000101,0.000058,0.000023,8.600000e-06,25.452503,...,25.499011,25.057325,24.504658,24.204211,87.700993,87.565880,87.660270,86.359595,81.563863,69.874200
1,1100015,2010-02-01,201002,0.000126,0.000099,0.000070,0.000025,0.000009,5.400000e-06,25.577318,...,25.070354,24.449709,23.873474,24.796041,88.718914,87.520821,87.454519,82.695120,70.338089,58.425858
2,1100015,2010-03-01,201003,0.000093,0.000059,0.000023,0.000009,0.000007,9.700000e-06,25.445710,...,24.442533,24.299101,25.030957,26.863161,87.758768,86.756887,81.366066,71.612738,60.323017,49.655235
3,1100015,2010-04-01,201004,0.000050,0.000022,0.000009,0.000003,0.000010,3.440000e-05,25.099890,...,24.166243,24.989621,27.304428,27.275026,85.654669,79.517193,69.389973,54.380982,48.631807,62.649872
4,1100015,2010-05-01,201005,0.000023,0.000007,0.000003,0.000007,0.000031,6.000000e-05,24.828948,...,25.252248,27.554440,27.805172,26.713236,80.604723,68.658103,50.846026,44.553284,59.072057,76.333297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1085950,5300108,2025-11-01,202511,0.000092,0.000106,0.000094,0.000090,0.000080,3.130000e-05,23.728494,...,22.699973,22.841146,22.567486,22.444757,73.231927,77.970517,77.428952,77.911010,79.199977,72.145321
1085951,5300108,2025-12-01,202512,0.000103,0.000088,0.000083,0.000071,0.000029,4.000000e-06,23.018151,...,22.922236,22.728722,22.336149,22.041078,76.114829,75.405680,76.944523,78.433866,72.262318,58.325962
1085952,5300108,2026-01-01,202601,0.000078,0.000084,0.000077,0.000032,0.000004,9.000000e-07,23.509768,...,22.734097,22.349373,21.963827,21.832059,72.391511,77.284455,79.491457,73.360449,58.925943,49.934165
1085953,5300108,2026-02-01,202602,0.000080,0.000075,0.000033,0.000003,0.000001,1.100000e-06,22.944970,...,22.361765,22.060199,21.413330,21.476499,78.424987,79.691840,74.330286,59.780818,51.019061,44.119279


In [27]:
# we are doing the same as we did for the climate df but now with the specificities of forecast - including the columns it has

forecast = forecast.merge(
    gdf[["geocode", "area", 'uf', 'uf_code']],
    on="geocode",
    how="left")

group_cols = ["year_month", "uf", "uf_code"]

value_cols = [c for c in forecast.columns if c not in non_climate]

weights = forecast["area"] / forecast.groupby(group_cols)["area"].transform("sum")

weighted = forecast[value_cols].mul(weights, axis=0)

forecast = (
    weighted
    .groupby([forecast["year_month"], forecast["uf"], forecast["uf_code"]])
    .sum()
    .reset_index()
)

In [28]:
forecast = forecast.drop(columns=["uf"])

In [29]:
# we now add the forecast to climate. A bit strange because the climate is on a weekly level and the forecast is on a monthly level
# so the first week of month 01 will have the same value for the forecasts as the 4th week of month 01
# We can discuss if we should do this in a different way
climate_uf = climate_uf.merge(forecast,on=["uf_code", "year_month"],how="left")

In [30]:
# Merge everything into uf_week

uf_week = uf_week.merge(
    climate_uf,
    on=["epiweek", "uf_code"],
    how="left")

In [31]:
# Getting the uf_code for each gecode
pop = pop.merge(
    gdf[["geocode", 'uf_code']],
    on="geocode",
    how="left")


In [32]:
#calculate pop of each uf
pop = (pop.groupby(["year", "uf_code"], as_index=False)["population"].sum())

In [33]:
# As we dont have data for the pop of 2026 we will use as of 2025
pop_2026 = pop.loc[pop["year"] == 2025].copy()

pop_2026["year"] = 2026

pop = pd.concat([pop, pop_2026], ignore_index=True)

In [34]:
# get a year variable on the uf_week to merge
uf_week["year"] = uf_week["epiweek"].astype(str).str[:4].astype("int64")

In [35]:
# Population is on a yearly level and obv has the same issue as the above
uf_week = uf_week.merge(
    pop[["uf_code", "year", "population"]],
    on=["uf_code", "year"],
    how="left")

OCEAN MERGING



In [36]:
# Ocean now arrives Sunday-aligned with a correct `epiweek` column and (currently) no missing weeks.
# The old Monday/Tuesday shifting logic is obsolete: on Sunday dates `weekday + 1` subtracts 7 days,
# pushing every row back a full week, and the date-based remerge then collided with the existing
# `epiweek` column (epiweek_x correct / epiweek_y wrong). So we drop the shift entirely.
ocean["date"] = pd.to_datetime(ocean["date"]).dt.normalize()
ocean = ocean[ocean["date"] >= pd.to_datetime(dengue["date"].min())]

cols = ["enso", "iod", "pdo"]

# Robust against any *future* gaps: reindex onto a complete weekly (Sun) grid and interpolate,
# then rebuild epiweek from dengue's authoritative date->epiweek map (no shift, no collision).
full_weeks = pd.date_range(ocean["date"].min(), ocean["date"].max(), freq="W-SUN")
ocean = ocean.set_index("date").reindex(full_weeks).sort_index()
ocean.index.name = "date"
ocean[cols] = ocean[cols].interpolate(method="linear", limit_direction="both")

ocean = ocean.drop(columns=["epiweek"]).reset_index()

dengue_map = dengue[["date", "epiweek"]].drop_duplicates()
dengue_map["date"] = pd.to_datetime(dengue_map["date"])
ocean = ocean.merge(dengue_map, on="date", how="left")

In [37]:
# merging to the uf_week
# the values of enso, iod and pdo will be the same for all the uf withi the same week
uf_week = uf_week.merge(
    ocean[["epiweek", "enso", "iod", "pdo"]],
    on="epiweek",
    how="left")

In [38]:
# With env will have to do something similar to climate ( weighted average) but before will hae to convert the biome and the koppen to one hot
# and then calculate the share of each and then average
env = env.merge(
    gdf[["geocode", "area",]],
    on="geocode",
    how="left")


In [39]:
env = pd.get_dummies(env, columns=["koppen", "biome"])

In [40]:
koppen_cols = [c for c in env.columns if c.startswith("koppen_")]
biome_cols = [c for c in env.columns if c.startswith("biome_")]

In [41]:
uf_total = env.groupby("uf_code")["area"].transform("sum")
weights = env["area"] / uf_total

In [42]:
env[koppen_cols + biome_cols] = env[koppen_cols + biome_cols].multiply(weights, axis=0)

env = env.groupby("uf_code", as_index=False).agg({**{c: "sum" for c in koppen_cols + biome_cols}})

In [43]:
env

,uf_code,koppen_Af,koppen_Am,koppen_As,koppen_Aw,koppen_BSh,koppen_Cfa,koppen_Cfb,koppen_Cwa,koppen_Cwb,biome_Amazônia,biome_Caatinga,biome_Cerrado,biome_Mata Atlântica,biome_Pampa,biome_Pantanal
0,11,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,12,0.679781,0.320219,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,13,0.808426,0.191574,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,14,0.441503,0.558497,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,15,0.215584,0.731150,0.000000,0.053266,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,16,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,17,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.088061,0.000000,0.911939,0.000000,0.000000,0.000000
7,21,0.000000,0.152814,0.094613,0.752573,0.000000,0.000000,0.000000,0.000000,0.000000,0.347432,0.000000,0.652568,0.000000,0.000000,0.000000
8,22,0.000000,0.000000,0.198324,0.607580,0.194096,0.000000,0.000000,0.000000,0.000000,0.000000,0.462749,0.537251,0.000000,0.000000,0.000000
9,23,0.000000,0.000000,0.612356,0.000000,0.387644,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000


In [44]:
# Not sure if this is usefull because it is very static
uf_week = uf_week.merge(
    env,
    on="uf_code",
    how="left")

In [45]:
uf_week = uf_week.rename(columns={"uf": "state"})

In [46]:
uf_week.cases.min()

0

In [47]:
uf_week = uf_week.merge(
    shape_regional,
    on="uf_code",
    how="left")

In [56]:
# DONT RUN THIS WITHOUT THINKING
uf_week.to_csv("../data_catalog/full_dataset.csv", index=False)